# BB84 / Ekert91 Demonstration — With Attacker

This notebook demonstrates the BB84 protocol **with an attacker**, usually called Eve.

The assignment asks us to show that:
- Alice and Bob can attempt to generate a key,
- Eve tries to obtain information about the key,
- Eve's measurement disrupts some qubits,
- Alice and Bob detect the attack using an error-rate threshold.

Although this file is named `Ekert91-Attacker.ipynb`, the implementation below follows the **BB84 protocol** required in the assignment brief.


In [1]:
# Install the required packages.
# Run this cell first if Qiskit is not already installed in your environment.

%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=89f39fd97a6838aac75da779dd63adda90e9a1977c7ae755030b3d24b36df933
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
# Imports used for the BB84 simulation.
# Important: We do NOT import or use Python's random module.
# All random bits, bases, and attack decisions are generated by quantum measurement.

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from IPython.display import display, Markdown

simulator = AerSimulator()


## Helper functions

Random values are generated by measuring a qubit prepared in equal superposition:

\[
\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)
\]

This follows the assignment instruction not to use Python's standard random number generator.


In [4]:
def quantum_random_bit():
    """
    Generate one random classical bit using quantum measurement.

    A Hadamard gate creates an equal superposition.
    Measuring the qubit gives 0 or 1 with approximately equal probability.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    result = simulator.run(qc, shots=1, memory=True).result()
    return int(result.get_memory()[0])


def quantum_random_bits(length):
    """Generate a list of random bits using quantum measurement only."""
    return [quantum_random_bit() for _ in range(length)]


def basis_name(basis):
    """Convert basis value into a readable label."""
    return "Z/+" if basis == 0 else "X/x"


def bit_string(bits):
    """Convert a list of bits into a compact string."""
    return "".join(str(bit) for bit in bits)


## Step 1 — Alice generates random bits and bases

Alice prepares the original BB84 qubits. Her bits and bases are private.


In [5]:
# Number of qubits to send.
# A larger value makes the attack easier to detect statistically.
N = 80

alice_bits = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

prepared_qubits = list(zip(alice_bits, alice_bases))

print("Alice bits:  ", bit_string(alice_bits))
print("Alice bases: ", [basis_name(b) for b in alice_bases])


Alice bits:   10111001100000011101000000000010101111101101000001110000010011111101111110000001
Alice bases:  ['X/x', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'X/x', 'X/x', 'X/x', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+']


## Step 2 — Eve intercepts the quantum channel

Eve performs an **intercept-resend attack**:

1. Eve chooses a random basis for each qubit.
2. Eve measures the qubit.
3. Eve resends a new qubit to Bob based on her measured bit and basis.

If Eve chooses the wrong basis, she may disturb the qubit. This disruption is what allows Alice and Bob to detect her.


In [6]:
# Eve attacks every qubit in this demonstration.
# To attack only some qubits, change ATTACK_ALL to False.
ATTACK_ALL = True

eve_bases = quantum_random_bits(N)
eve_results = []
qubits_sent_to_bob = []

for i in range(N):
    alice_bit = alice_bits[i]
    alice_basis = alice_bases[i]
    eve_basis = eve_bases[i]

    if ATTACK_ALL:
        if eve_basis == alice_basis:
            # Eve chose the correct basis, so she reads Alice's bit correctly.
            eve_bit = alice_bit
        else:
            # Eve chose the wrong basis, so her measurement result is random.
            eve_bit = quantum_random_bit()

        # Eve resends a new qubit according to her measured result and basis.
        qubits_sent_to_bob.append((eve_bit, eve_basis))
        eve_results.append(eve_bit)
    else:
        # No attack on this qubit: forward Alice's original qubit unchanged.
        qubits_sent_to_bob.append((alice_bit, alice_basis))
        eve_results.append(None)

print("Eve bases:")
print([basis_name(b) for b in eve_bases])
print("Eve measured bits:")
print(bit_string([b if b is not None else 0 for b in eve_results]))


Eve bases:
['X/x', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+']
Eve measured bits:
10011010100000101110111000000000000110111101000011111001010001011101011110010001


## Step 3 — Bob generates random measurement bases

Bob does not know that Eve has interfered. He chooses his measurement bases independently.


In [7]:
bob_bases = quantum_random_bits(N)

print("Bob bases:")
print([basis_name(b) for b in bob_bases])


Bob bases:
['Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'X/x', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'X/x', 'X/x', 'Z/+', 'Z/+', 'Z/+', 'Z/+', 'X/x', 'Z/+', 'Z/+']


## Step 4 — Bob measures the qubits received from Eve

Bob measures the qubits that arrive at his side. If Eve has disturbed some qubits, Bob's results may differ from Alice's original bits even when Bob and Alice used the same basis.


In [8]:
bob_results = []

for i in range(N):
    sent_bit, sent_basis = qubits_sent_to_bob[i]
    bob_basis = bob_bases[i]

    if bob_basis == sent_basis:
        # Bob uses the same basis as the resent qubit.
        measured_bit = sent_bit
    else:
        # Bob uses a different basis, so the outcome is random.
        measured_bit = quantum_random_bit()

    bob_results.append(measured_bit)

print("Bob results:")
print(bit_string(bob_results))


Bob results:
11001001100000101010111100111000100010101001010011110011000010011100011111011001


## Step 5 — Alice and Bob compare bases publicly

Alice and Bob reveal only their basis choices. They keep the positions where Alice's and Bob's bases match.

Even after basis sifting, Eve's wrong-basis measurements may create errors in the sifted key.


In [9]:
matching_positions = [
    i for i in range(N)
    if alice_bases[i] == bob_bases[i]
]

alice_sifted_key = [alice_bits[i] for i in matching_positions]
bob_sifted_key = [bob_results[i] for i in matching_positions]

print("Matching positions:", matching_positions)
print("Alice sifted key:", bit_string(alice_sifted_key))
print("Bob sifted key:  ", bit_string(bob_sifted_key))
print("Sifted key length:", len(alice_sifted_key))


Matching positions: [6, 7, 8, 9, 10, 11, 12, 13, 17, 24, 28, 29, 32, 33, 35, 39, 40, 41, 42, 43, 47, 49, 50, 52, 54, 55, 56, 58, 60, 64, 66, 69, 70, 71, 72, 74, 76, 79]
Alice sifted key: 01100000100010101101011000001101111001
Bob sifted key:   01100000001010001001011011001101111011
Sifted key length: 38


## Step 6 — Attack detection using an error threshold

Alice and Bob compare some sifted key bits to estimate the error rate.

For a simple demonstration, this notebook compares the whole sifted key. In a real system, the compared bits would be discarded and only the unrevealed bits would be used as the final secret key.

A typical intercept-resend attack introduces noticeable errors, often around 25% among the sifted positions.


In [10]:
errors = sum(
    1 for a, b in zip(alice_sifted_key, bob_sifted_key)
    if a != b
)

error_rate = errors / len(alice_sifted_key) if alice_sifted_key else 0

threshold = 0.20

print("Errors:", errors)
print("Error rate:", round(error_rate, 3))
print("Detection threshold:", threshold)

if error_rate > threshold:
    print("Attack detected! The key should be rejected.")
else:
    print("No attack detected. The key may be accepted, but more test bits may be needed.")


Errors: 7
Error rate: 0.184
Detection threshold: 0.2
No attack detected. The key may be accepted, but more test bits may be needed.


## Final conclusion

The attacker notebook demonstrates that Eve's intercept-resend attack disturbs the quantum states.  
When Alice and Bob compare their sifted key values, the error rate increases.  
If the error rate is above the chosen threshold, Alice and Bob reject the key and report that an attack was detected.
